<h3 style="font-family: Times New Roman"><strong>Laboratory Activity 6: CNN Architecture Implementation in PyTorch</strong></h3>

<p style="font-family:Times New Roman; text-align:justify; font-size:15px">
<b>Objective:</b> Convert the given CNN architecture diagram into a PyTorch CNN architecture.
</p>

# Laboratory Activity 6: CNN Architecture Implementation in PyTorch

<h4 style="font-family: Times New Roman"><strong>Architecture Diagram</strong></h4>

![CNN architecture diagram](attachment:diagram.jpg)

<h4 style="font-family: Times New Roman"><strong>Import Libraries</strong></h4>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

<h4 style="font-family: Times New Roman"><strong>Computing the Unknown Shapes</strong></h4>

<p style="font-family:Times New Roman; text-align:justify; font-size:15px">
The diagram leaves some shapes as <tt>?</tt>. These are computed using the output size formula for convolution and pooling layers:
</p>

$$\text{Output Size} = \left\lfloor \frac{W - F + 2P}{S} \right\rfloor + 1$$

<p style="font-family:Times New Roman; font-size:15px">where $W$ is the input size, $F$ the kernel size, $P$ the padding, and $S$ the stride.</p>

In [2]:
def calc_out(w, f, s, p):
    """
    Calculate output shape of a matrix after a convolution or pooling.
    The results are only applicable for square kernels and images only.
    """
    print(f'Output Shape: {(w - f + 2 * p) // s + 1}')

In [3]:
# Conv1: kernel=3, stride=1, padding=1
calc_out(28, 3, 1, 1)

Output Shape: 28


In [4]:
# MaxPool1: kernel=2, stride=2, padding=1
calc_out(28, 2, 2, 1)

Output Shape: 15


In [5]:
# Conv2, Conv3, Conv4: kernel=3, stride=1, padding=1
calc_out(15, 3, 1, 1)

Output Shape: 15


In [6]:
# MaxPool2: kernel=2, stride=2, padding=0
calc_out(15, 2, 2, 0)

Output Shape: 7


In [7]:
# Flatten: channels x height x width
256 * 7 * 7

12544

<p style="font-family:Times New Roman; text-align:justify; font-size:15px">
<strong>Summary of the shapes (per image):</strong>
</p>

| Layer | Output shape |
|:---|:---|
| Input | (1, 28, 28) |
| Conv1 + ReLU | (32, 28, 28) |
| MaxPool1 (padding=1) | (32, 15, 15) |
| Conv2 + ReLU | (64, 15, 15) |
| Conv3 + ReLU | (128, 15, 15) |
| Conv4 + ReLU | (256, 15, 15) |
| MaxPool2 (padding=0) | (256, 7, 7) |
| Dropout (p=0.2) | (256, 7, 7) |
| Flatten | (12544) |
| FCN1 + ReLU | (1000) |
| FCN2 + ReLU | (500) |
| FCN3 + Softmax | (number of classes) |

<h4 style="font-family: Times New Roman"><strong>PyTorch Implementation</strong></h4>

In [8]:
class QuickDrawCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=(3,3), stride=1, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=(2,2), stride=2, padding=1)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=(3,3), stride=1, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=(3,3), stride=1, padding=1)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=(3,3), stride=1, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=(2,2), stride=2, padding=0)

        self.dropout = nn.Dropout(p=0.2)

        self.fcn1 = nn.Linear(256*7*7, 1000)
        self.fcn2 = nn.Linear(1000, 500)
        self.fcn3 = nn.Linear(500, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)

        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = self.pool2(x)

        x = self.dropout(x)
        x = x.view(-1, 256*7*7)

        x = F.relu(self.fcn1(x))
        x = F.relu(self.fcn2(x))
        x = F.softmax(self.fcn3(x), dim=1)

        return x

<h4 style="font-family: Times New Roman"><strong>Instantiate the Model</strong></h4>

In [9]:
model = QuickDrawCNN(num_classes=10)
model

QuickDrawCNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=(2, 2), stride=2, padding=1, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv4): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=(2, 2), stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.2, inplace=False)
  (fcn1): Linear(in_features=12544, out_features=1000, bias=True)
  (fcn2): Linear(in_features=1000, out_features=500, bias=True)
  (fcn3): Linear(in_features=500, out_features=10, bias=True)
)

<h4 style="font-family: Times New Roman"><strong>Verifying the Layer Output Shapes</strong></h4>

<p style="font-family:Times New Roman; text-align:justify; font-size:15px">
A dummy batch of 32 images of size <tt>(1, 28, 28)</tt> (the batch size in the diagram) is passed through the model. A forward hook prints the output shape of every layer so it can be compared with the diagram and the table above.
</p>

In [10]:
def show_shape(name):
    def hook(module, inp, out):
        print(f"{name:8} -> {tuple(out.shape)}")
    return hook

for name, layer in model.named_children():
    layer.register_forward_hook(show_shape(name))

dummy = torch.randn(32, 1, 28, 28)
out = model(dummy)
print("output   ->", tuple(out.shape))

conv1    -> (32, 32, 28, 28)
pool1    -> (32, 32, 15, 15)
conv2    -> (32, 64, 15, 15)
conv3    -> (32, 128, 15, 15)
conv4    -> (32, 256, 15, 15)
pool2    -> (32, 256, 7, 7)
dropout  -> (32, 256, 7, 7)
fcn1     -> (32, 1000)
fcn2     -> (32, 500)
fcn3     -> (32, 10)
output   -> (32, 10)


<h4 style="font-family: Times New Roman"><strong>Model Parameters</strong></h4>

In [11]:
def count_parameters(model):
    params = [p.numel() for p in model.parameters() if p.requires_grad]
    for item in params:
        print(f'{item:>9}')
    print(f'_________\n{sum(params):>9}')

count_parameters(model)

      288
       32
    18432
       64
    73728
      128
   294912
      256
 12544000
     1000
   500000
      500
     5000
       10
_________
 13438350


<p style="font-family:Times New Roman; text-align:justify; font-size:15px">
Model Parameters: <br>
$(1\times32\times3\times3)+32+(32\times64\times3\times3)+64+(64\times128\times3\times3)+128+(128\times256\times3\times3)+256+(12544\times1000)+1000+(1000\times500)+500+(500\times10)+10 = 13,438,350$
</p>

<h4 style="font-family: Times New Roman"><strong>Discussion</strong></h4>

<p style="font-family:Times New Roman; text-align:justify; font-size:15px">
The diagram was translated layer by layer into a <tt>torch.nn.Module</tt>. All convolution layers use a <tt>3×3</tt> kernel with stride 1 and padding 1, so they preserve the spatial size of their input while the number of channels increases from 1 to 32, 64, 128, and finally 256. The spatial size is reduced only by the pooling layers. <tt>MaxPool1</tt> uses padding 1, which turns the <tt>28×28</tt> feature map into <tt>15×15</tt> instead of <tt>14×14</tt>, and <tt>MaxPool2</tt> (no padding) reduces it to <tt>7×7</tt>. These values determine the input size of <tt>FCN1</tt>: the flattened feature map has <tt>256 × 7 × 7 = 12544</tt> values per image.
</p>

<p style="font-family:Times New Roman; text-align:justify; font-size:15px">
Dropout with <tt>p=0.2</tt> is applied after the second pooling layer to reduce overfitting, and is active only in training mode. The fully connected part reduces the features to <tt>1000</tt>, then <tt>500</tt>, and the last layer outputs one value per class, converted to probabilities by softmax. Most of the parameters (about 12.5 million of the 13.4 million) are in <tt>FCN1</tt>, so the fully connected layers make up most of the model size, while the convolution layers, which extract the features, are comparatively small.
</p>